# Задание 1: RAG Pipeline
## Naive RAG + Advanced RAG

Проект по построению RAG-пайплайна для ответов на вопросы по годовым отчётам казахстанских компаний (КТЖ и Матен Петролеум).

**Все в одном ноутбуке** — без внешних модулей, без проблем с импортами.

In [26]:
# Установка зависимостей (выполнить один раз). --user если Access denied
# DocLing, LlamaParse, Qdrant (векторная БД), RAGAS (Задание 2)
!pip install --user docling llama-parse sentence-transformers langchain langchain-openai langchain-text-splitters rank-bm25 FlagEmbedding python-dotenv tiktoken qdrant-client nest_asyncio ragas -q


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Конфигурация и импорты

In [27]:
# КОНФИГУРАЦИЯ
from pathlib import Path
import os

# Пути
DATA_DIR = Path("./data")
assert DATA_DIR.exists(), "Создайте папку data/ и поместите ktj.pdf и matnp_2024_rus.pdf"

# API ключ (из .env или переменной окружения)
from dotenv import load_dotenv
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
LLAMAPARSE_API_KEY = os.getenv("LLAMAPARSE_API_KEY", "") or os.getenv("LLAMA_CLOUD_API_KEY", "")

# Параметры пайплайна
CONFIG = {
    "parser": "llamaparse",  # "llamaparse" после сравнения, если таблицы лучше
    "chunk_size": 1024,
    "chunk_overlap": 200,
    "embedding_model": "paraphrase-multilingual-mpnet-base-v2",
    "llm_model": "gpt-4o-mini",
    "top_k": 5,
    "rerank_top_n": 3,
    "alpha": 0.5,  # Hybrid Search: 0=BM25, 1=Vector
}

print("Конфигурация загружена.")

Конфигурация загружена.


In [28]:
# ИМПОРТЫ
import numpy as np
import tiktoken
from sentence_transformers import SentenceTransformer
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

print("Импорты загружены.")

Импорты загружены.


## 0. Сравнение парсеров (DocLing vs LlamaParse)

**Цель:** сравнить качество парсинга таблиц. Оба результата сохраняются — визуально сравни в `data/`:
- `*_parsed_docling.md` — DocLing (локальный)
- `*_parsed_llamaparse.md` — LlamaParse (облачный API)

**Роадмап после сравнения:**
1. Выбрать лучший парсер по таблицам
2. Эмбеддинги (SentenceTransformer)
3. Naive RAG (Dense retrieval, Qdrant)
4. Advanced RAG (Hybrid, Reranker, Query Rewriting)

In [29]:
# Сравнение парсеров: DocLing и LlamaParse. Оба результата сохраняются.
def parse_pdf_docling(pdf_path):
    from docling.document_converter import DocumentConverter
    converter = DocumentConverter()
    result = converter.convert(str(pdf_path))
    return result.document.export_to_markdown()

def parse_pdf_llamaparse(pdf_path, api_key=None):
    import nest_asyncio
    nest_asyncio.apply()
    from llama_parse import LlamaParse
    key = api_key or LLAMAPARSE_API_KEY or os.getenv("LLAMA_CLOUD_API_KEY")
    if not key:
        raise ValueError("Нужен LLAMAPARSE_API_KEY или LLAMA_CLOUD_API_KEY в .env")
    parser = LlamaParse(api_key=key, result_type="markdown", verbose=True, language="ru")
    docs = parser.load_data(str(pdf_path))
    return "\n\n".join(d.text for d in docs) if docs else ""

# 1. DocLing — используем существующие *_parsed.md, если есть (без повторного парсинга)
print("=== DocLing ===")
docs_docling = {}
for name, fname in [("ktj", "ktj.pdf"), ("matnp", "matnp_2024_rus.pdf")]:
    out_docling = DATA_DIR / f"{name}_parsed_docling.md"
    existing = DATA_DIR / f"{name}_parsed.md"  # старый DocLing вывод
    if out_docling.exists():
        docs_docling[name] = out_docling.read_text(encoding="utf-8")
        print(f"  Загружен из кэша: {out_docling}")
    elif existing.exists():
        docs_docling[name] = existing.read_text(encoding="utf-8")
        out_docling.write_text(docs_docling[name], encoding="utf-8")
        print(f"  Скопирован {existing} → {out_docling}")
    else:
        print(f"Парсинг {fname}...")
        docs_docling[name] = parse_pdf_docling(DATA_DIR / fname)
        out_docling.write_text(docs_docling[name], encoding="utf-8")
        print(f"  Сохранён: {out_docling}")

# 2. LlamaParse (облачный)
print("\n=== LlamaParse ===")
docs_lp = {}
for name, fname in [("ktj", "ktj.pdf"), ("matnp", "matnp_2024_rus.pdf")]:
    path = DATA_DIR / fname
    if path.exists():
        print(f"Парсинг {fname}...")
        docs_lp[name] = parse_pdf_llamaparse(path)
        out_lp = DATA_DIR / f"{name}_parsed_llamaparse.md"
        out_main = DATA_DIR / f"{name}_parsed.md"
        out_lp.write_text(docs_lp[name], encoding="utf-8")
        out_main.write_text(docs_lp[name], encoding="utf-8")
        print(f"  Сохранён: {out_lp}")
        print(f"  Сохранён (основной): {out_main}")

print("\nГотово. Сравни таблицы в data/*_parsed_docling.md и data/*_parsed_llamaparse.md")

=== DocLing ===
  Загружен из кэша: data\ktj_parsed_docling.md
  Загружен из кэша: data\matnp_parsed_docling.md

=== LlamaParse ===
Парсинг ktj.pdf...


2026-02-28 23:45:52,973 - INFO - HTTP Request: POST https://api.cloud.llamaindex.ai/api/parsing/upload "HTTP/1.1 200 OK"


Started parsing the file under job_id 47a8a05b-8ebf-4b95-8887-e32697f359fe


2026-02-28 23:45:54,244 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/47a8a05b-8ebf-4b95-8887-e32697f359fe "HTTP/1.1 200 OK"
2026-02-28 23:45:56,501 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/47a8a05b-8ebf-4b95-8887-e32697f359fe "HTTP/1.1 200 OK"
2026-02-28 23:45:59,771 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/47a8a05b-8ebf-4b95-8887-e32697f359fe "HTTP/1.1 200 OK"
2026-02-28 23:46:04,029 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/47a8a05b-8ebf-4b95-8887-e32697f359fe "HTTP/1.1 200 OK"
2026-02-28 23:46:09,272 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/47a8a05b-8ebf-4b95-8887-e32697f359fe "HTTP/1.1 200 OK"
2026-02-28 23:46:14,941 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/47a8a05b-8ebf-4b95-8887-e32697f359fe "HTTP/1.1 200 OK"
2026-02-28 23:46:20,667 - INFO - HTTP Request: GET https://api.cloud.llamain

  Сохранён: data\ktj_parsed_llamaparse.md
  Сохранён (основной): data\ktj_parsed.md
Парсинг matnp_2024_rus.pdf...


2026-02-28 23:46:30,292 - INFO - HTTP Request: POST https://api.cloud.llamaindex.ai/api/parsing/upload "HTTP/1.1 200 OK"


Started parsing the file under job_id 36c6515e-b8d4-4495-b9cc-4ac3ae6d25ef


2026-02-28 23:46:31,566 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/36c6515e-b8d4-4495-b9cc-4ac3ae6d25ef "HTTP/1.1 200 OK"
2026-02-28 23:46:33,823 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/36c6515e-b8d4-4495-b9cc-4ac3ae6d25ef "HTTP/1.1 200 OK"
2026-02-28 23:46:37,088 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/36c6515e-b8d4-4495-b9cc-4ac3ae6d25ef "HTTP/1.1 200 OK"
2026-02-28 23:46:37,450 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/36c6515e-b8d4-4495-b9cc-4ac3ae6d25ef/result/markdown "HTTP/1.1 200 OK"


  Сохранён: data\matnp_parsed_llamaparse.md
  Сохранён (основной): data\matnp_parsed.md

Готово. Сравни таблицы в data/*_parsed_docling.md и data/*_parsed_llamaparse.md


## 2. Функции пайплайна (всё в одном месте)

In [30]:
# ---------- Парсинг PDF ----------
# CONFIG["parser"] = "docling" (по умолч.) или "llamaparse" после сравнения
def parse_pdf_docling(pdf_path):
    from docling.document_converter import DocumentConverter
    converter = DocumentConverter()
    result = converter.convert(str(pdf_path))
    return result.document.export_to_markdown()

def parse_pdf_llamaparse(pdf_path):
    import nest_asyncio
    nest_asyncio.apply()
    from llama_parse import LlamaParse
    key = LLAMAPARSE_API_KEY or os.getenv("LLAMA_CLOUD_API_KEY")
    if not key:
        raise ValueError("Добавь LLAMAPARSE_API_KEY в .env")
    parser = LlamaParse(api_key=key, result_type="markdown", verbose=False, language="ru")
    docs = parser.load_data(str(pdf_path))
    return "\n\n".join(d.text for d in docs) if docs else ""

def parse_pdf(pdf_path):
    return parse_pdf_docling(pdf_path)  # алиас

def parse_all_pdfs(data_dir, parser=None):
    parser = parser or CONFIG.get("parser", "docling")
    fn = parse_pdf_llamaparse if parser == "llamaparse" else parse_pdf_docling
    docs = {}
    for name, fname in [("ktj", "ktj.pdf"), ("matnp", "matnp_2024_rus.pdf")]:
        path = data_dir / fname
        if not path.exists():
            raise FileNotFoundError(f"PDF не найден: {path}")
        docs[name] = fn(path)
    return docs

# ---------- Чанкинг ----------
def naive_chunk(text, chunk_size=1024, overlap=200):
    enc = tiktoken.get_encoding("cl100k_base")
    tokens = enc.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + chunk_size, len(tokens))
        chunk_text = enc.decode(tokens[start:end])
        chunks.append({"text": chunk_text})
        start = end - overlap if end < len(tokens) else len(tokens)
    return chunks

def recursive_chunk(text, chunk_size=1024, overlap=200):
    splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        encoding_name="cl100k_base", chunk_size=chunk_size, chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    return [{"text": c} for c in splitter.split_text(text)]

def markdown_to_blocks(md_text):
    blocks = []
    lines = md_text.split("\n")
    i = 0
    while i < len(lines):
        line = lines[i]
        if line.strip().startswith("#"):
            blocks.append({"type": "heading", "text": line.strip()})
            i += 1
        elif "|" in line and line.strip().startswith("|"):
            tbl = [line]
            i += 1
            while i < len(lines) and "|" in lines[i]:
                tbl.append(lines[i]); i += 1
            blocks.append({"type": "table", "text": "\n".join(tbl)})
        elif line.strip():
            para = [line]
            i += 1
            while i < len(lines) and lines[i].strip() and not lines[i].strip().startswith("#") and "|" not in lines[i]:
                para.append(lines[i]); i += 1
            blocks.append({"type": "paragraph", "text": "\n".join(para)})
        else:
            i += 1
    return blocks

def layout_aware_chunk(blocks, max_tokens=1024):
    enc = tiktoken.get_encoding("cl100k_base")
    chunks, current, current_tok, header = [], [], 0, ""
    for b in blocks:
        t = b.get("text", "")
        tok = len(enc.encode(t))
        if b.get("type") == "heading": header = t
        if current_tok + tok > max_tokens and current:
            chunks.append({"text": "\n\n".join(current)})
            current, current_tok = [], 0
        current.append(t)
        current_tok += tok
    if current:
        chunks.append({"text": "\n\n".join(current)})
    return chunks

print("Парсинг и чанкинг готовы.")

Парсинг и чанкинг готовы.


In [31]:
# ---------- Embeddings ----------
def get_embeddings(model, texts, prefix=""):
    """Эмбеддинги с нормализацией. prefix для E5: 'passage' или 'query'."""
    if prefix and "e5" in str(model).lower():
        texts = [f"{prefix}: {t}" for t in texts] if isinstance(texts, list) else [f"{prefix}: {texts}"]
    embs = model.encode(texts if isinstance(texts, list) else [texts])
    embs = np.array(embs)
    norm = np.linalg.norm(embs, axis=1, keepdims=True)
    norm[norm == 0] = 1
    return (embs / norm).tolist()

# ---------- Retrieval ----------
def dense_retrieval(q_emb, chunk_embs, chunk_texts, top_k=5):
    q = np.array(q_emb, dtype=np.float32)
    embs = np.array(chunk_embs, dtype=np.float32)
    scores = np.dot(embs, q)
    idx = np.argsort(scores)[::-1][:top_k]
    return [(chunk_texts[i], float(scores[i])) for i in idx]

def bm25_retrieval(query, chunk_texts, top_k=5):
    tok = lambda t: t.lower().split()
    bm25 = BM25Okapi([tok(t) for t in chunk_texts])
    scores = bm25.get_scores(tok(query))
    idx = np.argsort(scores)[::-1][:top_k]
    return [(chunk_texts[i], float(scores[i])) for i in idx]

def hybrid_search(query, q_emb, chunk_embs, chunk_texts, alpha=0.5, top_k=10, k=60):
    if alpha >= 1.0: return dense_retrieval(q_emb, chunk_embs, chunk_texts, top_k)
    if alpha <= 0.0: return bm25_retrieval(query, chunk_texts, top_k)
    d = dense_retrieval(q_emb, chunk_embs, chunk_texts, top_k * 3)
    b = bm25_retrieval(query, chunk_texts, top_k * 3)
    rv = {t: i+1 for i, (t, _) in enumerate(d)}
    rb = {t: i+1 for i, (t, _) in enumerate(b)}
    all_t = set(rv) | set(rb)
    scores = {t: alpha/(k+rv.get(t,999)) + (1-alpha)/(k+rb.get(t,999)) for t in all_t}
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

print("Embeddings и Retrieval готовы.")

Embeddings и Retrieval готовы.


In [32]:
# ---------- Reranker ----------
def get_reranker():
    from FlagEmbedding import FlagReranker
    try:
        import torch
        fp16 = torch.cuda.is_available()
    except:
        fp16 = False
    return FlagReranker("BAAI/bge-reranker-v2-m3", use_fp16=fp16)

def rerank_docs(reranker, query, docs, top_n=3):
    if not docs: return []
    pairs = [[query, d] for d in docs]
    scores = reranker.compute_score(pairs, normalize=True)
    if isinstance(scores, float): scores = [scores]
    idx = np.argsort(scores)[::-1][:top_n]
    return [(docs[i], scores[i]) for i in idx]

# ---------- Query Rewriting ----------
def rewrite_query(llm, q):
    """Pre-Retrieval: переформулировка запроса для лучшего поиска."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Переформулируй вопрос для поиска по отчётам КТЖ и Матен Петролеум. Добавь ключевые термины. Ответь только запросом."),
        ("human", "{q}")
    ])
    r = (prompt | llm).invoke({"q": q})
    return r.content.strip() if hasattr(r, "content") else str(r).strip()

print("Reranker и Query Rewriting готовы.")

Reranker и Query Rewriting готовы.


## 3. Часть A: Naive RAG (20 баллов)

- Парсинг PDF (DocLing)
- Naive Chunking: 1024 токена, overlap 200
- Embeddings: paraphrase-multilingual-mpnet-base-v2
- Векторная БД: Qdrant (локальное хранение)
- Dense Retrieval (косинусное сходство)
- LLM: GPT-4o-mini

In [33]:
# Загрузка данных и построение Naive RAG
import pickle
cfg = CONFIG
CACHE_PATH = DATA_DIR / "rag_cache.pkl"

emb_model = SentenceTransformer(cfg["embedding_model"])

if CACHE_PATH.exists():
    print("Загрузка из кэша (без парсинга)...")
    with open(CACHE_PATH, "rb") as f:
        cache = pickle.load(f)
    chunks = cache["chunks"]
    chunk_texts = cache["chunk_texts"]
    chunk_embs = cache["chunk_embs"]
else:
    print("Парсинг PDF...")
    docs = parse_all_pdfs(DATA_DIR)
    for name, md in docs.items():
        path = DATA_DIR / f"{name}_parsed.md"
        path.write_text(md, encoding="utf-8")
        print(f"  Сохранён Markdown: {path}")
    full_text = "\n\n".join(docs.values())
    print("Чанкинг...")
    chunks = naive_chunk(full_text, cfg["chunk_size"], cfg["chunk_overlap"])
    chunk_texts = [c["text"] for c in chunks]
    print("Эмбеддинги...")
    chunk_embs = get_embeddings(emb_model, chunk_texts)
    with open(CACHE_PATH, "wb") as f:
        pickle.dump({"chunks": chunks, "chunk_texts": chunk_texts, "chunk_embs": chunk_embs}, f)
    print("Кэш сохранён.")

llm = ChatOpenAI(model=cfg["llm_model"], temperature=0, api_key=OPENAI_API_KEY)
SYSTEM = "Ты помощник по годовым отчётам КТЖ и Матен Петролеум. Отвечай ТОЛЬКО по контексту. Не выдумывай."

def naive_query(question):
    q_emb = get_embeddings(emb_model, question, prefix="")[0]
    results = dense_retrieval(q_emb, chunk_embs, chunk_texts, top_k=cfg["top_k"])
    context = "\n\n---\n\n".join([r[0] for r in results])
    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM),
        ("human", "Контекст:\n{ctx}\n\nВопрос: {q}\n\nОтвет:")
    ])
    r = (prompt | llm).invoke({"ctx": context, "q": question})
    return r.content if hasattr(r, "content") else str(r)

print(f"Naive RAG готов. Чанков: {len(chunks)}")

2026-02-28 23:46:38,100 - INFO - Use pytorch device_name: cpu
2026-02-28 23:46:38,100 - INFO - Load pretrained SentenceTransformer: paraphrase-multilingual-mpnet-base-v2


Парсинг PDF...


2026-02-28 23:46:49,851 - INFO - HTTP Request: POST https://api.cloud.llamaindex.ai/api/parsing/upload "HTTP/1.1 200 OK"
2026-02-28 23:46:51,077 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/d572a655-d73f-4b06-b5f5-b0052d75a90b "HTTP/1.1 200 OK"
2026-02-28 23:46:53,291 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/d572a655-d73f-4b06-b5f5-b0052d75a90b "HTTP/1.1 200 OK"
2026-02-28 23:46:56,612 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/d572a655-d73f-4b06-b5f5-b0052d75a90b "HTTP/1.1 200 OK"
2026-02-28 23:47:00,906 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/d572a655-d73f-4b06-b5f5-b0052d75a90b "HTTP/1.1 200 OK"
2026-02-28 23:47:06,104 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/d572a655-d73f-4b06-b5f5-b0052d75a90b "HTTP/1.1 200 OK"
2026-02-28 23:47:11,842 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/d572a655-d

  Сохранён Markdown: data\ktj_parsed.md
  Сохранён Markdown: data\matnp_parsed.md
Чанкинг...
Эмбеддинги...


Batches: 100%|██████████| 2/2 [00:14<00:00,  7.04s/it]

Кэш сохранён.
Naive RAG готов. Чанков: 60


In [34]:
# Загрузка в Qdrant (использует chunk_embs и chunk_texts из ячейки выше — без повторного парсинга!)
import gc
try:
    if "qdrant_client" in globals() and qdrant_client is not None:
        qdrant_client._client._close()
except Exception:
    pass
gc.collect()

QDRANT_PATH = "./qdrant_data"
COLLECTION_NAME = "rag_chunks"
try:
    qdrant_client = QdrantClient(path=QDRANT_PATH)
except RuntimeError as e:
    if "already accessed" in str(e).lower():
        QDRANT_PATH = "./qdrant_data_2"  # запасной путь, если папка занята
        qdrant_client = QdrantClient(path=QDRANT_PATH)
        print(f"Используется {QDRANT_PATH} (основная папка была занята)")
    else:
        raise
try:
    qdrant_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass
qdrant_client.create_collection(COLLECTION_NAME, vectors_config=VectorParams(size=len(chunk_embs[0]), distance=Distance.COSINE))
qdrant_client.upsert(COLLECTION_NAME, points=[PointStruct(id=i, vector=emb, payload={"text": txt}) for i, (emb, txt) in enumerate(zip(chunk_embs, chunk_texts))])

def naive_query(question):
    q_emb = get_embeddings(emb_model, question, prefix="")[0]
    resp = qdrant_client.query_points(collection_name=COLLECTION_NAME, query=q_emb, limit=cfg["top_k"])
    results = [(p.payload.get("text", ""), p.score) for p in resp.points]
    context = "\n\n---\n\n".join([r[0] for r in results])
    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM),
        ("human", "Контекст:\n{ctx}\n\nВопрос: {q}\n\nОтвет:")
    ])
    r = (prompt | llm).invoke({"ctx": context, "q": question})
    return r.content if hasattr(r, "content") else str(r)

print("Чанки загружены в Qdrant. naive_query теперь использует векторную БД.")

Чанки загружены в Qdrant. naive_query теперь использует векторную БД.


### Тестирование Naive RAG

In [35]:
test_questions = [
    "Каков был доход от основной деятельности АО «НК «ҚТЖ» в 2024 году?",
    "Какой объем реализации нефти был зафиксирован у «Матен Петролеум» в 2024 году?",
    "В каком городе базируется АО «Матен Петролеум» согласно отчету?",
    "Какие международные стандарты ISO внедрены в системе управления КТЖ?",
    "Какие крупные инфраструктурные объекты планирует ввести в эксплуатацию КТЖ в 2025 году?",
]

print("=== Naive RAG: Результаты ===")
for q in test_questions:
    print(f"Q: {q}")
    print(f"A: {naive_query(q)}")
    print("-" * 60)

=== Naive RAG: Результаты ===
Q: Каков был доход от основной деятельности АО «НК «ҚТЖ» в 2024 году?


Batches: 100%|██████████| 1/1 [00:00<00:00,  9.38it/s]
2026-02-28 23:47:44,633 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


A: Консолидированная выручка АО «НК «ҚТЖ» в 2024 году составила 2 163,9 млрд тенге, что на 11,9% больше по сравнению с 2023 годом.
------------------------------------------------------------
Q: Какой объем реализации нефти был зафиксирован у «Матен Петролеум» в 2024 году?


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.56it/s]
2026-02-28 23:47:46,511 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


A: Объем реализации нефти у «Матен Петролеум» в 2024 году составил 676 651 тонн.
------------------------------------------------------------
Q: В каком городе базируется АО «Матен Петролеум» согласно отчету?


...

A: АО «Матен Петролеум» базируется в городе Атырау, Республика Казахстан.
------------------------------------------------------------
Q: Какие международные стандарты ISO внедрены в системе управления КТЖ?


Batches: 100%|██████████| 1/1 [00:00<00:00,  7.46it/s]
2026-02-28 23:47:53,143 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


A: В предоставленном контексте не указаны конкретные международные стандарты ISO, внедренные в системе управления АО «НК «ҚТЖ». Для получения этой информации необходимо обратиться к дополнительным источникам или разделам отчета, которые могут содержать данные о внедрении стандартов ISO.
------------------------------------------------------------
Q: Какие крупные инфраструктурные объекты планирует ввести в эксплуатацию КТЖ в 2025 году?


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.28it/s]
2026-02-28 23:47:55,693 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


A: В 2025 году КТЖ планирует ввести в эксплуатацию следующие крупные инфраструктурные объекты:

- Второй ж/д путь на участке «Достык – Мойынты»;
- Железнодорожную линию в обход ст. Алматы.
------------------------------------------------------------


### Типичные ошибки Naive RAG

1. **Таблицы разрываются** — цифры из разных строк путаются, контекст теряется при чанкинге.
2. **Точные названия не находятся** — «Достык – Мойынты», ISO-стандарты и т.п. теряются при чисто семантическом поиске.
3. **Числа и даты** — dense retrieval хуже находит точные совпадения (например, «676 651 тонн»), BM25 справляется лучше.
4. **Фрагментация контекста** — один ответ размазан по нескольким чанкам, LLM «склеивает» с ошибками.
5. **Повторный retrieval** — один и тот же фрагмент попадает в top‑k несколько раз в разных формулировках.

**Advanced RAG** в следующих ячейках помогает устранить эти проблемы: Hybrid Search (Vector + BM25), Reranker, Layout-Aware чанкинг.

## 4. Часть B: Advanced RAG (30 баллов)

- 3 стратегии чанкинга: Naive, Recursive, Layout-Aware
- Hybrid Search (Semantic + BM25, RRF)
- Reranking (Cross-Encoder -> little llm for ranking ~300-500 mln prm) -> top-k
- Pre-Retrieval: Query Rewriting

In [36]:
# Advanced RAG с полным набором улучшений
def build_advanced_rag(chunk_strategy="recursive", use_reranker=True, use_query_rewrite=True):
    cfg = CONFIG
    docs = parse_all_pdfs(DATA_DIR)
    for name, md in docs.items():
        path = DATA_DIR / f"{name}_parsed.md"
        path.write_text(md, encoding="utf-8")
        print(f"  Markdown сохранён: {path}")
    full_text = "\n\n".join(docs.values())
    
    if chunk_strategy == "naive":
        chunks = naive_chunk(full_text, cfg["chunk_size"], cfg["chunk_overlap"])
    elif chunk_strategy == "recursive":
        chunks = recursive_chunk(full_text, cfg["chunk_size"], cfg["chunk_overlap"])
    else:
        blocks = markdown_to_blocks(full_text)
        chunks = layout_aware_chunk(blocks)
    
    chunk_texts = [c["text"] for c in chunks]
    emb_model = SentenceTransformer(cfg["embedding_model"])
    chunk_embs = get_embeddings(emb_model, chunk_texts)
    llm = ChatOpenAI(model=cfg["llm_model"], temperature=0, api_key=OPENAI_API_KEY)
    reranker = get_reranker() if use_reranker else None
    
    def query(question):
        search_q = rewrite_query(llm, question) if use_query_rewrite else question
        q_emb = get_embeddings(emb_model, search_q, prefix="")[0]
        results = hybrid_search(search_q, q_emb, chunk_embs, chunk_texts, alpha=cfg["alpha"], top_k=10)
        if use_reranker and reranker:
            docs_for_rerank = [r[0] for r in results]
            results = rerank_docs(reranker, question, docs_for_rerank, top_n=cfg["rerank_top_n"])
        context = "\n\n---\n\n".join([r[0] for r in results])
        prompt = ChatPromptTemplate.from_messages([
            ("system", SYSTEM),
            ("human", "Контекст:\n{ctx}\n\nВопрос: {q}\n\nОтвет:")
        ])
        r = (prompt | llm).invoke({"ctx": context, "q": question})
        return r.content if hasattr(r, "content") else str(r)
    
    return {"query": query, "chunks": chunks}

adv_pipeline = build_advanced_rag(chunk_strategy="recursive", use_reranker=True, use_query_rewrite=True)
print(f"Advanced RAG готов. Чанков: {len(adv_pipeline['chunks'])}")

2026-02-28 23:47:58,805 - INFO - HTTP Request: POST https://api.cloud.llamaindex.ai/api/parsing/upload "HTTP/1.1 200 OK"
2026-02-28 23:48:00,067 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/91344dbc-7e03-459a-9dc0-ee5f2bcae9e4 "HTTP/1.1 200 OK"
2026-02-28 23:48:02,337 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/91344dbc-7e03-459a-9dc0-ee5f2bcae9e4 "HTTP/1.1 200 OK"
2026-02-28 23:48:05,611 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/91344dbc-7e03-459a-9dc0-ee5f2bcae9e4 "HTTP/1.1 200 OK"
2026-02-28 23:48:09,875 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/91344dbc-7e03-459a-9dc0-ee5f2bcae9e4 "HTTP/1.1 200 OK"
2026-02-28 23:48:15,686 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/91344dbc-7e03-459a-9dc0-ee5f2bcae9e4 "HTTP/1.1 200 OK"
2026-02-28 23:48:21,298 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/91344dbc-7

  Markdown сохранён: data\ktj_parsed.md
  Markdown сохранён: data\matnp_parsed.md


Batches: 100%|██████████| 2/2 [00:14<00:00,  7.41s/it]


Advanced RAG готов. Чанков: 64


### Сравнение 3 стратегий чанкинга

In [37]:
strategies = ["naive", "recursive", "layout_aware"]
pipelines = {}
for s in strategies:
    pipelines[s] = build_advanced_rag(chunk_strategy=s, use_reranker=True, use_query_rewrite=True)
    print(f"{s}: {len(pipelines[s]['chunks'])} чанков")

2026-02-28 23:49:12,659 - INFO - HTTP Request: POST https://api.cloud.llamaindex.ai/api/parsing/upload "HTTP/1.1 200 OK"
2026-02-28 23:49:13,931 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/f31ac3e7-5e44-4bc5-8467-873e1d32a2f3 "HTTP/1.1 200 OK"
2026-02-28 23:49:16,188 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/f31ac3e7-5e44-4bc5-8467-873e1d32a2f3 "HTTP/1.1 200 OK"
2026-02-28 23:49:19,456 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/f31ac3e7-5e44-4bc5-8467-873e1d32a2f3 "HTTP/1.1 200 OK"
2026-02-28 23:49:23,690 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/f31ac3e7-5e44-4bc5-8467-873e1d32a2f3 "HTTP/1.1 200 OK"
2026-02-28 23:49:29,436 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/f31ac3e7-5e44-4bc5-8467-873e1d32a2f3 "HTTP/1.1 200 OK"
2026-02-28 23:49:35,021 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/f31ac3e7-5

  Markdown сохранён: data\ktj_parsed.md
  Markdown сохранён: data\matnp_parsed.md


Batches: 100%|██████████| 2/2 [00:13<00:00,  6.94s/it]


naive: 60 чанков


2026-02-28 23:50:13,054 - INFO - HTTP Request: POST https://api.cloud.llamaindex.ai/api/parsing/upload "HTTP/1.1 200 OK"
2026-02-28 23:50:14,280 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/66cbf439-c2ea-4a20-b11d-379b7127de5c "HTTP/1.1 200 OK"
2026-02-28 23:50:16,528 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/66cbf439-c2ea-4a20-b11d-379b7127de5c "HTTP/1.1 200 OK"
2026-02-28 23:50:19,780 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/66cbf439-c2ea-4a20-b11d-379b7127de5c "HTTP/1.1 200 OK"
2026-02-28 23:50:24,038 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/66cbf439-c2ea-4a20-b11d-379b7127de5c "HTTP/1.1 200 OK"
2026-02-28 23:50:29,285 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/66cbf439-c2ea-4a20-b11d-379b7127de5c "HTTP/1.1 200 OK"
2026-02-28 23:50:34,515 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/66cbf439-c

...

recursive: 64 чанков


2026-02-28 23:51:15,799 - INFO - HTTP Request: POST https://api.cloud.llamaindex.ai/api/parsing/upload "HTTP/1.1 200 OK"
2026-02-28 23:51:17,185 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/7e308c04-0f2d-4437-9dfb-8acd4bf37dda "HTTP/1.1 200 OK"
2026-02-28 23:51:19,434 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/7e308c04-0f2d-4437-9dfb-8acd4bf37dda "HTTP/1.1 200 OK"
2026-02-28 23:51:22,767 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/7e308c04-0f2d-4437-9dfb-8acd4bf37dda "HTTP/1.1 200 OK"
2026-02-28 23:51:27,104 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/7e308c04-0f2d-4437-9dfb-8acd4bf37dda "HTTP/1.1 200 OK"
2026-02-28 23:51:32,328 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/7e308c04-0f2d-4437-9dfb-8acd4bf37dda "HTTP/1.1 200 OK"
2026-02-28 23:51:37,594 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/7e308c04-0

  Markdown сохранён: data\ktj_parsed.md
  Markdown сохранён: data\matnp_parsed.md


Batches: 100%|██████████| 2/2 [00:12<00:00,  6.43s/it]


layout_aware: 54 чанков


### Тест Advanced RAG

In [38]:
# 2 сложных (данные из таблиц) + 2 простых (факты из текста)
questions = [
    "Каков был доход от основной деятельности АО «НК «ҚТЖ» в 2024 году?",  # сложно: таблица
    "Какой объём реализации нефти был зафиксирован у «Матен Петролеум» в 2024 году?",  # сложно: таблица
    "В каком городе базируется АО «Матен Петролеум» согласно отчету?",  # просто
    "Какие крупные инфраструктурные объекты планирует ввести в эксплуатацию КТЖ в 2025 году?",  # просто
]
for q in questions:
    print("Вопрос:", q)
    print("Ответ:", adv_pipeline["query"](q))
    print("-" * 60)

Вопрос: Каков был доход от основной деятельности АО «НК «ҚТЖ» в 2024 году?


2026-02-28 23:52:09,490 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Batches: 100%|██████████| 1/1 [00:00<00:00,  5.03it/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
2026-02-28 23:53:27,467 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Ответ: Доход от основной деятельности АО «НК «ҚТЖ» в 2024 году составил 2 163,9 млрд тенге.
------------------------------------------------------------
Вопрос: Какой объём реализации нефти был зафиксирован у «Матен Петролеум» в 2024 году?


2026-02-28 23:53:28,418 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Batches: 100%|██████████| 1/1 [00:00<00:00,  5.35it/s]
2026-02-28 23:54:38,508 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Ответ: Объем реализации нефти у «Матен Петролеум» в 2024 году составил 676 651 тонн.
------------------------------------------------------------
Вопрос: В каком городе базируется АО «Матен Петролеум» согласно отчету?


2026-02-28 23:54:40,579 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Batches: 100%|██████████| 1/1 [00:00<00:00,  4.57it/s]
2026-02-28 23:55:51,884 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Ответ: АО «Матен Петролеум» базируется в городе Атырау.
------------------------------------------------------------
Вопрос: Какие крупные инфраструктурные объекты планирует ввести в эксплуатацию КТЖ в 2025 году?


2026-02-28 23:55:52,971 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Batches: 100%|██████████| 1/1 [00:00<00:00,  4.78it/s]
2026-02-28 23:57:06,091 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Ответ: В 2025 году КТЖ планирует ввести в эксплуатацию следующие крупные инфраструктурные объекты:

- Второй ж/д путь на участке «Достык – Мойынты»;
- Железнодорожную линию в обход ст. Алматы;
- Запуск терминалов в порту Актау, в городе Алматы, на станциях Селятино (Россия) и Свислочь (Беларусь).
------------------------------------------------------------


### Reranking: до и после

In [42]:
adv_with = build_advanced_rag(use_reranker=True)
adv_without = build_advanced_rag(use_reranker=False)
# Вопрос, где hybrid search может вернуть шум (Матен — «стандарты» без ISO), а reranker отдаст приоритет чанку КТЖ с конкретными номерами ISO
q = "Какой клиент приносит наибольшую долю выручки и какой у него процент?"
print("С reranker:", adv_with["query"](q))
print("Без reranker:", adv_without["query"](q))

2026-03-01 00:08:57,418 - INFO - HTTP Request: POST https://api.cloud.llamaindex.ai/api/parsing/upload "HTTP/1.1 200 OK"
2026-03-01 00:08:58,694 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/8cff7749-49a3-438e-85d6-fa0bc13f310b "HTTP/1.1 200 OK"
2026-03-01 00:09:01,009 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/8cff7749-49a3-438e-85d6-fa0bc13f310b "HTTP/1.1 200 OK"
2026-03-01 00:09:04,257 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/8cff7749-49a3-438e-85d6-fa0bc13f310b "HTTP/1.1 200 OK"
2026-03-01 00:09:08,531 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/8cff7749-49a3-438e-85d6-fa0bc13f310b "HTTP/1.1 200 OK"
2026-03-01 00:09:14,111 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/8cff7749-49a3-438e-85d6-fa0bc13f310b "HTTP/1.1 200 OK"
2026-03-01 00:09:19,765 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/8cff7749-4

  Markdown сохранён: data\ktj_parsed.md
  Markdown сохранён: data\matnp_parsed.md


2026-03-01 00:09:28,679 - INFO - Use pytorch device_name: cpu
2026-03-01 00:09:28,684 - INFO - Load pretrained SentenceTransformer: paraphrase-multilingual-mpnet-base-v2
Batches: 100%|██████████| 2/2 [00:15<00:00,  7.93s/it]
2026-03-01 00:10:01,787 - INFO - HTTP Request: POST https://api.cloud.llamaindex.ai/api/parsing/upload "HTTP/1.1 200 OK"
2026-03-01 00:10:03,060 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/5c54ffc1-f8f1-4bff-8541-7a876c6be8d7 "HTTP/1.1 200 OK"
2026-03-01 00:10:05,315 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/5c54ffc1-f8f1-4bff-8541-7a876c6be8d7 "HTTP/1.1 200 OK"
2026-03-01 00:10:08,639 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/5c54ffc1-f8f1-4bff-8541-7a876c6be8d7 "HTTP/1.1 200 OK"
2026-03-01 00:10:12,869 - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/5c54ffc1-f8f1-4bff-8541-7a876c6be8d7 "HTTP/1.1 200 OK"
2026-03-01 00:10:18,123 - INFO - HTTP 

  Markdown сохранён: data\ktj_parsed.md
  Markdown сохранён: data\matnp_parsed.md


Batches: 100%|██████████| 2/2 [00:15<00:00,  7.57s/it]
2026-03-01 00:10:58,183 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Batches: 100%|██████████| 1/1 [00:00<00:00,  4.53it/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
2026-03-01 00:12:10,680 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


С reranker: Наибольшую долю выручки приносит клиент Vitol Energy Trading SA, который составил 75% от общего дохода Группы в 2024 году.


2026-03-01 00:12:11,715 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Batches: 100%|██████████| 1/1 [00:00<00:00,  4.31it/s]
2026-03-01 00:12:14,055 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Без reranker: Наибольшую долю выручки приносит клиент Vitol Energy Trading SA, который составил 75% от общего дохода Группы в 2024 году.


### Query Rewriting (Pre-Retrieval)

In [40]:
llm = ChatOpenAI(model=CONFIG["llm_model"], temperature=0, api_key=OPENAI_API_KEY)
original = "Сколько заработала КТЖ?"
rewritten = rewrite_query(llm, original)
print("Оригинал:", original)
print("Переформулированный:", rewritten)

2026-03-01 00:00:28,769 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Оригинал: Сколько заработала КТЖ?
Переформулированный: Каковы финансовые результаты КТЖ за последний отчетный период?


## Итоги

- **Naive RAG**: базовый пайплайн, подвержен разрывам таблиц и потере точных совпадений
- **Advanced RAG**: Hybrid Search + Reranking + Query Rewriting существенно улучшают качество
- **Layout-Aware чанкинг**: сохраняет таблицы целиком, лучше для финансовых данных